## How to use functions from the sdg-tagger

In [ ]:
#Installs all python packages listed in the requirements file. You can skip this step if you have already done this in your envirionment.
%pip install -r requirements.txt

### Imports used in the entire file. Run this cell first!

In [ ]:
import json 
from rich import print_json #This is used for colouring when printing the output

### Search for all targets in a specific goal, and get the results in json format
Function: `search_all_targets_in_goal`

In [ ]:
# -------- Import the function we want to use from src --------
from src import search_in_text_for_one_sdg 

# ------- Define the SDG number and input text --------
SDG_NR = 2
TEXT = "This is a title about extreme poverty and ending extreme hunger in least developed countries."
#include_indexed_results = False # Set this to True if you want to also see the index (position) in the text where the search terms were found

# ------------------ Searching ----------------------
results = search_in_text_for_one_sdg(sdg_nr=SDG_NR, input_text=TEXT)

# --------------- Printing the results ---------------
print_json(json.dumps(results, indent=4)) # With coloured output (works best with light colour themes in your editor)
#print(json.dumps(results, indent=4)) # Use this to have the normal uncoloured output


### Search all goals on one text
Function: `search_all_goals`

In [ ]:
from src import search_all_goals

text = "This is a title about extreme poverty and ending extreme hunger in least developed countries."
results = search_all_goals(text)
print(json.dumps(results, indent=4))
#print_json(json.dumps(results, indent=4)) # Use this if you want the coloured output

### Trigger all country category searches
Function: `all_country_searches`

In [ ]:
from src import run_all_country_searches

text = 'This should be true for LDC, LMIC and LDS because I mention Burkina Faso.'
results_boolean = run_all_country_searches(input_text=text)
print(json.dumps(results_boolean, indent=4))
#print_json(json.dumps(results_boolean, indent=4)) # Use this if you want the coloured output

### Run search for all sdgs on an entire dataframe
Function: `dataframe_search`

In [ ]:
# Enter you file path, the SDGs you want to search for, and the columns in which you want to search
FILE_NAME = r"C:\xxx\xxx\TESTdata_titles-abstracts-SDGlabels_2020-2023.csv"
SDGS = [1,2]
VIEW = ["LMIC", "sdg01_01", "sdg01_02", "sdg02_01"]
COLUMNS = ['result_title', 'abstract'] # searchable columns
EXTRA_COLUMNS = ["result_id"] #columns for viewing
INCLUDE_FALSE_VALUES = False 

In [ ]:
from src import dataframe_search
import pandas as pd

# Read the data from your file
data_raw = pd.read_csv(FILE_NAME, sep=",")
data_raw = data_raw[:2]
data = data_raw[COLUMNS]

# Perform the search
result_df = data_raw[COLUMNS]
for index in range(len(COLUMNS)):
    text_col = COLUMNS[index]
    print(f'Searching in {text_col}')
    col_result = dataframe_search(data[[text_col]], sdg_list=SDGS, text_column=text_col)
    col_result = col_result.drop(text_col, axis=1)

    if index==0:
        original_columns = [col for col in col_result.columns if col not in COLUMNS]
        
    col_result = col_result.add_suffix(f'_{text_col}')
    result_df = pd.merge(result_df, col_result, left_index=True, right_index=True)
    prev_col = text_col

# Create a new, interleaved column order
interleaved_cols = []
for col in original_columns:
    for text_col in COLUMNS:
        interleaved_cols.append(col + f'_{text_col}')

cols_to_view = [col for col in interleaved_cols if any(v in col for v in VIEW)]
final_cols = COLUMNS + EXTRA_COLUMNS + cols_to_view
final_df = pd.concat([data_raw[EXTRA_COLUMNS + COLUMNS], result_df[cols_to_view]], axis=1)

#final_df = result_df[COLUMNS + interleaved_cols]

if not INCLUDE_FALSE_VALUES:
    # Creates a mask to filter only on rows that contains some true values
    falsy_mask = (final_df[final_cols].fillna(False) == False).all(axis=1)
    final_df = final_df[~falsy_mask]

# Colours all values that are not nan or False
# Only style the SDG columns for coloring, and text columns for width
text_cols_to_style = [col for col in COLUMNS if col in final_df.columns]

def color_true_green(val):
    if (pd.notna(val)) and (val is not False):
        return f'background-color: green'
    
styled_df = final_df.style.map(color_true_green, subset=cols_to_view).set_properties(
        subset=text_cols_to_style,   # your long-text column
        **{
            "min-width": "600px",
            "max-width": "600px",
            "white-space": "normal"
        })
styled_df

Saving the styled results to excel

In [ ]:
file_path = 'styled_output.xlsx'
try:
    styled_df.to_excel(file_path, engine='openpyxl', index=False)
    print(f"Successfully saved styled DataFrame to {file_path}")
except ImportError as e:
    print(f"Error: {e}. Please ensure 'openpyxl' or 'xlsxwriter' is installed.")
except Exception as e:
    print(f"An error occurred: {e}")

### Getting help to use a function correct:
Use `help(<function name>)` to get info about the function you are using

In [ ]:
from src import run_all_country_searches

help(run_all_country_searches)